In [2]:
import json
from pathlib import Path
from datetime import datetime

from ultralytics import YOLO
import torch


In [3]:
# ---------------------------------------------------------------------------
# CONFIGURACIÓN
# ---------------------------------------------------------------------------
DATA_YAML = Path(r"D:\Publicaciones\SIMEE\detection_dataset\data.yaml")
OUTPUT_DIR = Path(r"D:\Publicaciones\SIMEE\yolov8_run")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Misma semilla única usada en el entrenamiento de InceptionV3, por consistencia
SEED = 42

In [4]:
# ---------------------------------------------------------------------------
# AJUSTE AUTOMÁTICO AL HARDWARE — detecta la VRAM disponible y adapta
# el tamaño del modelo, el batch y la resolución de imagen. Esto evita
# errores de "CUDA out of memory" en GPUs con poca memoria (p.ej. 4GB).
# El paper reporta YOLOv8m a 640x640 y batch=8; esos son los valores que
# se usan si detecta suficiente VRAM (>=8GB). Con menos, se reduce en
# cascada para que el entrenamiento simplemente funcione.
# ---------------------------------------------------------------------------
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    gpu_name = torch.cuda.get_device_properties(0).name
    DEVICE = 0  # primera GPU visible
    print(f"[hardware] GPU detectada: {gpu_name} ({vram_gb:.1f} GB VRAM) -> device={DEVICE}")
else:
    vram_gb = 0
    gpu_name = None
    DEVICE = "cpu"
    print("=" * 70)
    print("[hardware] ADVERTENCIA: no se detectó GPU. Se entrenará en CPU.")
    print("Esto puede tardar DÍAS en vez de horas para 100 épocas.")
    print("Verifica tu instalación de PyTorch con CUDA antes de continuar.")
    print("=" * 70)

if vram_gb >= 8:
    MODEL_ARCH = "yolov8m.pt"   # arquitectura reportada en el paper
    BATCH_SIZE = 8
    STAGE1_IMG_SIZE = 640
elif vram_gb >= 6:
    MODEL_ARCH = "yolov8m.pt"
    BATCH_SIZE = 4
    STAGE1_IMG_SIZE = 640
elif vram_gb >= 3:
    # Caso GTX 1650 Ti (4GB) y similares: YOLOv8m a batch=8/640px no entra en memoria.
    MODEL_ARCH = "yolov8s.pt"   # variante más liviana que 'm'
    BATCH_SIZE = 4
    STAGE1_IMG_SIZE = 512
    print("[hardware] VRAM limitada (<6GB): se usa YOLOv8s en vez de YOLOv8m, "
          "batch=4 e imgsz=512 para evitar errores de memoria. "
          "Esto es una DESVIACIÓN respecto al paper original — documentar en Métodos.")
else:
    MODEL_ARCH = "yolov8n.pt"
    BATCH_SIZE = 4
    STAGE1_IMG_SIZE = 416
    print("[hardware] VRAM muy limitada o CPU: usando YOLOv8n con configuración mínima.")

# Valores reportados en el paper original (Sección II.D) para la Etapa 1
STAGE1_EPOCHS = 50
STAGE1_LR0 = 5e-4          # tal como reporta el paper (Adam, lr inicial 5x10^-4)

# Etapa 2 (fine-tuning): el paper dice "mismos parámetros, 50 épocas adicionales,
# LR reducido". El LR reducido exacto no se especifica; se asume 1/10 del inicial.
STAGE2_EPOCHS = 50
STAGE2_LR0 = 5e-5


[hardware] GPU detectada: NVIDIA GeForce GTX 1650 Ti (4.0 GB VRAM) -> device=0
[hardware] VRAM limitada (<6GB): se usa YOLOv8s en vez de YOLOv8m, batch=4 e imgsz=512 para evitar errores de memoria. Esto es una DESVIACIÓN respecto al paper original — documentar en Métodos.


In [5]:
# ---------------------------------------------------------------------------
# ETAPA 1 — entrenamiento inicial (equivalente al "initial training stage"
# del paper: 50 épocas, 640x640, Adam, warm-up de 3 épocas, cosine LR schedule)
# ---------------------------------------------------------------------------
model = YOLO(MODEL_ARCH)

print("\n" + "=" * 70)
print("CONFIGURACIÓN FINAL DE ENTRENAMIENTO")
print(f"  Dispositivo:   {DEVICE}")
print(f"  Modelo:        {MODEL_ARCH}")
print(f"  Batch size:    {BATCH_SIZE}")
print(f"  Tamaño imagen: {STAGE1_IMG_SIZE}")
print("=" * 70 + "\n")

print("=== ETAPA 1: entrenamiento inicial ===")
results_stage1 = model.train(
    data=str(DATA_YAML),
    epochs=STAGE1_EPOCHS,
    imgsz=STAGE1_IMG_SIZE,
    batch=BATCH_SIZE,
    optimizer="Adam",
    lr0=STAGE1_LR0,
    cos_lr=True,
    warmup_epochs=3,
    seed=SEED,
    patience=10,  # early stopping; el paper no especifica el valor exacto (asumido)
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name="stage1",
    exist_ok=True,
    # Augmentaciones equivalentes a las reportadas en el paper (flips, escala,
    # shear, Mosaic, MixUp, perturbaciones HSV como proxy de brillo/contraste)
    fliplr=0.5,
    flipud=0.5,
    translate=0.1,
    scale=0.5,
    shear=0.1,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
)

stage1_best = OUTPUT_DIR / "stage1" / "weights" / "best.pt"



CONFIGURACIÓN FINAL DE ENTRENAMIENTO
  Dispositivo:   0
  Modelo:        yolov8s.pt
  Batch size:    4
  Tamaño imagen: 512

=== ETAPA 1: entrenamiento inicial ===
Ultralytics 8.4.150  Python-3.10.21 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce GTX 1650 Ti, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\Publicaciones\SIMEE\detection_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=512, iou=0.7, keras=False, kobj=1.0, li

C:\Users\Nicolas\.conda\envs\entornoGPU2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
WARNING val: Slow image access detected (ping: 0.20.3 ms, read: 20.416.5 MB/s, size: 676.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning D:\Publicaciones\SIMEE\detection_dataset\labels\val.cache... 831 images, 360 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 831/831  0.0s
optimizer: Adam(lr=0.0005, momentum=0.937) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Plotting labels to D:\Publicaciones\SIMEE\yolov8_run\stage1\labels.jpg... 
Using 3716 train, 831 val images for fraction=1.0 at imgsz=512
Using 8 dataloader workers
Logging results to D:\Publicaciones\SIMEE\yolov8_run\stage1
Starting training for 50 epochs...

      Epoch    

In [6]:
# ---------------------------------------------------------------------------
# ETAPA 2 — fine-tuning desde los pesos de la Etapa 1, LR reducido
# ---------------------------------------------------------------------------
print("\n=== ETAPA 2: fine-tuning ===")
model_stage2 = YOLO(str(stage1_best))
results_stage2 = model_stage2.train(
    data=str(DATA_YAML),
    epochs=STAGE2_EPOCHS,
    imgsz=STAGE1_IMG_SIZE,
    batch=BATCH_SIZE,
    optimizer="Adam",
    lr0=STAGE2_LR0,
    cos_lr=True,
    warmup_epochs=0,  # ya viene "calentado" de la Etapa 1
    seed=SEED,
    patience=10,
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name="stage2",
    exist_ok=True,
    fliplr=0.5,
    flipud=0.5,
    translate=0.1,
    scale=0.5,
    shear=0.1,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
)

stage2_best = OUTPUT_DIR / "stage2" / "weights" / "best.pt"


=== ETAPA 2: fine-tuning ===
Ultralytics 8.4.150  Python-3.10.21 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce GTX 1650 Ti, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\Publicaciones\SIMEE\detection_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=5e-05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=D:\Publicaciones\SIMEE\yolov8_run\stage1\we

In [7]:
# ---------------------------------------------------------------------------
# EVALUACIÓN FINAL en el TEST set (incluye ahora imágenes background)
# ---------------------------------------------------------------------------
print("\n=== Evaluación final en TEST ===")
final_model = YOLO(str(stage2_best))
test_metrics = final_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=STAGE1_IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name="test_eval",
    exist_ok=True,
    plots=True,  # genera automáticamente curvas P-R, F1-confidence, matriz de confusión
    save_json=True,  # guarda las predicciones detalladas por imagen
)


=== Evaluación final en TEST ===
Ultralytics 8.4.150  Python-3.10.21 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce GTX 1650 Ti, 4096MiB)
Model summary (fused): 72 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
WARNING val: Slow image access detected (ping: 7.77.6 ms, read: 35.98.5 MB/s, size: 536.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning D:\Publicaciones\SIMEE\detection_dataset\labels\test... 961 images, 360 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 961/961 104.0it/s 9.2s0.0s
val: New cache created: D:\Publicaciones\SIMEE\detection_dataset\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 241/241 15.4it/s 15.7s0.1s
                   all        961        650      0.795      0.205      0.227      0.103
                  mass        336        348       0.59      0.411      0.429      0.198
  

In [8]:
# ---------------------------------------------------------------------------
# Guardar metadatos de la corrida
# ---------------------------------------------------------------------------
run_metadata = {
    "timestamp": datetime.now().isoformat(),
    "seed": SEED,
    "gpu_name": gpu_name if vram_gb > 0 else None,
    "vram_gb": round(vram_gb, 1),
    "model_arch": MODEL_ARCH,
    "stage1_epochs_max": STAGE1_EPOCHS,
    "stage1_lr0": STAGE1_LR0,
    "stage2_epochs_max": STAGE2_EPOCHS,
    "stage2_lr0_assumed": STAGE2_LR0,
    "img_size": STAGE1_IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "classes": {0: "mass", 1: "calcification"},
    "test_metrics": {
        "precision": float(test_metrics.box.mp),
        "recall": float(test_metrics.box.mr),
        "map50": float(test_metrics.box.map50),
        "map50_95": float(test_metrics.box.map),
    },
}
with open(OUTPUT_DIR / "run_metadata_yolo.json", "w") as f:
    json.dump(run_metadata, f, indent=2)

print(f"\nResultados finales: {run_metadata['test_metrics']}")
print(f"Metadatos guardados en {OUTPUT_DIR / 'run_metadata_yolo.json'}")
print(f"Predicciones detalladas (save_json) y gráficos (plots) en {OUTPUT_DIR / 'test_eval'}")
print("(esas predicciones detalladas son la base para calcular FROC y el false-positive")
print(" rate en background sin reentrenar, como quedó pendiente en el paper)")



Resultados finales: {'precision': 0.7949383946574781, 'recall': 0.20545977011494254, 'map50': 0.22745134662296276, 'map50_95': 0.10278659579019138}
Metadatos guardados en D:\Publicaciones\SIMEE\yolov8_run\run_metadata_yolo.json
Predicciones detalladas (save_json) y gráficos (plots) en D:\Publicaciones\SIMEE\yolov8_run\test_eval
(esas predicciones detalladas son la base para calcular FROC y el false-positive
 rate en background sin reentrenar, como quedó pendiente en el paper)
